# Immune Data Preprocessing

In [14]:
import anndata as ad
import pandas as pd
import scanpy as sc

from pathlib import Path
import json

In [9]:
## Set data folder
CZI_DATA_PATH = Path("../data/cellxgene_data")
MODEL_PATH = Path("../models/scimmune")

In [6]:
## Load CZI immune data
filtered_immune_data = CZI_DATA_PATH / "immune_900K_filtered.h5ad"
adata_immune = sc.read_h5ad(filtered_immune_data)

/home/s5srinivasan/immune-foundational-model/.venv/lib64/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
# Axis 0 = cells
for field in adata_immune.obs.keys().tolist():
    if pd.api.types.is_categorical_dtype(adata_immune.obs[field]):
        adata_immune.obs[field] = adata_immune.obs[field].cat.remove_unused_categories() # remove unused categories



/tmp/ipykernel_83413/2266299424.py:3: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(adata_immune.obs[field]):


## Create new vocab with cell type tokens

In [17]:
cell_type_ids = adata_immune.obs.cell_type_ontology_term_id.unique() # extract unique cell type ids

vocab_file = MODEL_PATH / "vocab.json" # get vocab.json
updated_vocab_file = MODEL_PATH / "vocab_w_cell_type.json" # get vocab.json

# Load the existing vocab.json
with open(vocab_file, "r") as f:
    vocab = json.load(f)

# Extract unique cell type IDs and prepend "<cell_type"
new_tokens = [f"<cell_type={cell_type_id}>" for cell_type_id in cell_type_ids]

# Add new tokens to the vocab
for token in new_tokens:
    if token not in vocab:
        vocab[token] = len(vocab)  # Assign the next available index

# Save the updated vocab.json
with open(updated_vocab_file, "w") as f:
    json.dump(vocab, f, indent=2)